# Phase 3, experiment 10: giving recent time steps more weight

**Why this experiment:** The test period (steps 36 to 49) comes straight after the training period,
and the data changes over time. If the most recent training steps are more like the test period than
the early ones, it may help to let them count for more when the model is fitted.

**The one change in this experiment:** each training row gets a weight that depends on how old its
time step is:

`weight = decay ^ (last training step - row's time step)`

- Rows from the last training step get weight 1.
- Each step further back is multiplied by `decay` once more. With `decay = 0.9`, a row one step older
  gets 0.9, two steps older 0.81, and so on.
- `decay = 1.0` gives every row weight 1, which is the plain model.

LightGBM's `sample_weight` tells the model how much each row counts when it builds its trees. A row
with weight 0.5 counts half as much as a row with weight 1.

| decay | weight of a row 10 steps before the last training step | weight 20 steps before |
|---|---|---|
| 1.0 | 1.00 | 1.00 |
| 0.98 | 0.82 | 0.67 |
| 0.95 | 0.60 | 0.36 |
| 0.9 | 0.35 | 0.12 |
| 0.8 | 0.11 | 0.01 |

**Decision:** try recent-step weighting as the last experiment before finalising.

Model: default LightGBM on the 165 raw features. Validation: the three expanding-window folds. As in
notebooks 10 and 11, an option is worth keeping only if it raises the fold mean without lowering fold 2.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
raw_cols = [c for c in train.columns if c.startswith("feat_")]

labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

## 2. Score each decay value on the folds

In [ ]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]
fold_names = [f["name"] for f in FOLDS]

DECAYS = [1.0, 0.98, 0.95, 0.9, 0.8]   

def recency_weights(time_steps, last_step, decay):
    # Weight for each training row: decay to the power of how many steps
    steps_back = last_step - time_steps
    return decay ** steps_back

score_rows = []
for fold in FOLDS:
    tr = labeled[labeled["time_step"] <= fold["train_last"]]
    va = labeled[(labeled["time_step"] > fold["train_last"]) & (labeled["time_step"] <= fold["valid_last"])]

    for decay in DECAYS:
        weights = recency_weights(tr["time_step"], fold["train_last"], decay)

        model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
        model.fit(tr[raw_cols], tr["label"], sample_weight=weights)
        va_pred = model.predict_proba(va[raw_cols])[:, 1]

        row = {"decay": decay, "fold": fold["name"], "val_auc": roc_auc_score(va["label"], va_pred)}
        if fold["name"] == "fold 2":
            in_24 = (va["time_step"] == 24).values
            row["auc_step_24"] = roc_auc_score(va["label"][in_24], va_pred[in_24])
        score_rows.append(row)
    print("done:", fold["name"])

scores = pd.DataFrame(score_rows)

done: fold 1
done: fold 2
done: fold 3


## 3. Results

`change_mean` and `change_fold2` compare each decay with no weighting (`decay = 1.0`).

In [3]:
results = scores.pivot(index="decay", columns="fold", values="val_auc").loc[DECAYS]
results["mean"] = results[fold_names].mean(axis=1)
results["change_mean"] = results["mean"] - results.loc[1.0, "mean"]
results["change_fold2"] = results["fold 2"] - results.loc[1.0, "fold 2"]
results["auc_step_24"] = scores.dropna(subset=["auc_step_24"]).set_index("decay")["auc_step_24"]
results.round(4)

fold,fold 1,fold 2,fold 3,mean,change_mean,change_fold2,auc_step_24
decay,,,,,,,
1.00,0.9888,0.9572,0.9965,0.9808,0.0000,0.0000,0.7866
0.98,0.9910,0.9427,0.9968,0.9768,-0.0040,-0.0145,0.7427
0.95,0.9881,0.9482,0.9963,0.9775,-0.0033,-0.0090,0.7594
0.90,0.9932,0.9510,0.9961,0.9801,-0.0007,-0.0062,0.7656
0.80,0.9924,0.9504,0.9966,0.9798,-0.0010,-0.0068,0.7704


## 4. No submission

No decay value raised the fold mean without lowering fold 2, so no submission is made from this
notebook.

## Findings

- **Does recent-step weighting raise the fold mean without lowering fold 2?** No. Every decay value
  lowers the mean (by 0.0007 to 0.0040) and lowers fold 2 (by 0.0062 to 0.0145).
- **How do the folds respond as the decay gets stronger?** Fold 1 goes up for three of the four decay
  values (by up to 0.0044 at `decay = 0.9`; at `0.95` it goes down by 0.0007), fold 3 stays within
  0.0004 of the plain model, and fold 2 goes down every time. The effect does not grow steadily with the
  decay: the gentlest weighting (`0.98`) does the most damage to fold 2. This is the same pattern as the
  tuning experiment (notebook 10).
- **What happens on step 24?** It falls with every decay value, from 0.787 to between 0.743 and 0.770.
  This fits the earlier finding that step 24's problem is a kind of illicit transaction that is absent
  from the training steps, which giving the recent training steps more weight cannot supply.
